In [17]:
import http.client
import json

insights = ["plane", "trump", "bitcoin"]
# insights = ["bitcoin", ]

conn = http.client.HTTPSConnection("google.serper.dev")
payload = json.dumps([
    {
        "q": insight,
        "num": 10,
        "tbs": "qdr:d"
    } for insight in insights
])
headers = {
  'X-API-KEY': '39ef0015c9282897135dcf73ee553d994cfb895d',
  'Content-Type': 'application/json'
}

In [18]:
print(payload)

[{"q": "plane", "num": 10, "tbs": "qdr:d"}, {"q": "trump", "num": 10, "tbs": "qdr:d"}, {"q": "bitcoin", "num": 10, "tbs": "qdr:d"}]


In [19]:
conn.request("POST", "/news", payload, headers)
res = conn.getresponse()
data = res.read()
print(data.decode("utf-8"))

[{"searchParameters":{"q":"plane","type":"news","num":10,"tbs":"qdr:d","engine":"google"},"news":[{"title":"American Airlines Plane Crash Tragedy","link":"https://www.lcps.org/article/1997510","snippet":"Dear LCPS families and staff,. Our hearts are heavy as we process the devastating news of last night's tragic plane crash over the Potomac River involving...","date":"5 hours ago","source":"Loudoun County Public Schools","imageUrl":"https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTa0WRHYiNxrrLcuu5aTLfVhUPbMYgLYiNqE5cC7eIu6LAbJwm85MdJxG_c&usqp=CAI&s","position":1},{"title":"Pilot of American Airlines jet that crashed near Washington DC had Georgia ties","link":"https://www.fox5atlanta.com/news/pilot-american-airlines-jet-crashed-near-washington-dc-had-georgia-ties","snippet":"A family with ties to Georgia is grieving after learning their loved one was one of the pilots killed in the crash between a small American Airlines plane...","date":"5 hours ago","source":"FOX 5 Atlanta","im

In [20]:
# Parse the JSON data
parsed_data = json.loads(data.decode("utf-8"))

# Pretty print the data with indentation
print(json.dumps(parsed_data, indent=2))


[
  {
    "searchParameters": {
      "q": "plane",
      "type": "news",
      "num": 10,
      "tbs": "qdr:d",
      "engine": "google"
    },
    "news": [
      {
        "title": "American Airlines Plane Crash Tragedy",
        "link": "https://www.lcps.org/article/1997510",
        "snippet": "Dear LCPS families and staff,. Our hearts are heavy as we process the devastating news of last night's tragic plane crash over the Potomac River involving...",
        "date": "5 hours ago",
        "source": "Loudoun County Public Schools",
        "imageUrl": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTa0WRHYiNxrrLcuu5aTLfVhUPbMYgLYiNqE5cC7eIu6LAbJwm85MdJxG_c&usqp=CAI&s",
        "position": 1
      },
      {
        "title": "Pilot of American Airlines jet that crashed near Washington DC had Georgia ties",
        "link": "https://www.fox5atlanta.com/news/pilot-american-airlines-jet-crashed-near-washington-dc-had-georgia-ties",
        "snippet": "A family with ties to Georg

In [42]:
import requests
from bs4 import BeautifulSoup

def get_high_res_image(page_url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0",
            # Only request the HTML, not images/css/js
            "Accept": "text/html",
            # Allow compressed responses
            "Accept-Encoding": "gzip, deflate"
        }
        # Add timeout to avoid hanging on slow responses
        response = requests.get(page_url, headers=headers)
        
        # Parse only the head section where meta tags usually are
        head_content = response.text.split('</head>')[0] + '</head>'
        soup = BeautifulSoup(head_content, 'html.parser')

        # Try to find OpenGraph image first (most common and fastest check)
        og_image = soup.find("meta", property="og:image")
        if og_image and og_image.get("content"):
            return og_image["content"]

        # If no OG image, only then parse the full body
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Look for first image with srcset
        img_with_srcset = soup.find("img", srcset=True)
        if img_with_srcset:
            return img_with_srcset["srcset"].split(",")[-1].split(" ")[0]

        # Last resort: first image
        first_img = soup.find("img", src=True)
        return first_img["src"] if first_img else None

    except Exception:
        return None

In [43]:
import pandas as pd

# Initialize lists to store the data
titles = []
links = []
snippets = []
dates = []
sources = []
query_terms = []
image_urls = []

# Iterate through each query result
for i, query_result in enumerate(parsed_data):
    for article in query_result.get('news', []):
        titles.append(article.get('title', ''))
        links.append(article.get('link', ''))
        snippets.append(article.get('snippet', ''))
        dates.append(article.get('date', ''))
        sources.append(article.get('source', ''))
        query_terms.append(insights[i])
        image_urls.append(get_high_res_image(article.get('link', '')))

# Create the DataFrame
df = pd.DataFrame({
    # 'query_term': query_terms,
    'title': titles,
    'link': links,
    'snippet': snippets,
    'time_scraped': dates,
    'query_term': query_terms,
    'image_url': image_urls,
    'source': sources
})

# Display the first few rows
display(df)


,title,link,snippet,time_scraped,query_term,image_url,source
0,American Airlines Plane Crash Tragedy,https://www.lcps.org/article/1997510,"Dear LCPS families and staff,. Our hearts are ...",5 hours ago,plane,https://core-docs.s3.amazonaws.com/loudoun_cou...,Loudoun County Public Schools
1,Pilot of American Airlines jet that crashed ne...,https://www.fox5atlanta.com/news/pilot-america...,A family with ties to Georgia is grieving afte...,5 hours ago,plane,https://images.foxtv.com/static.fox5atlanta.co...,FOX 5 Atlanta
2,Here's what is known about the deadly collisio...,https://apnews.com/article/ronald-reagan-natio...,A jet with 60 passengers and four crew members...,14 minutes ago,plane,https://dims.apnews.com/dims4/default/0370ebd/...,AP News
3,D.C. plane crash live updates: Trump says ther...,https://www.nbcnews.com/news/us-news/live-blog...,The latest news and live updates on the plane ...,LIVE13 minutes ago,plane,https://media-cldnry.s-nbcnews.com/image/uploa...,NBC News
4,What we know about the American Airlines plane...,https://www.cbsnews.com/news/crash-reagan-nati...,A regional jet carrying 64 people collided wit...,1 hour ago,plane,https://assets3.cbsnewsstatic.com/hub/i/r/2025...,CBS News
5,Collision between helicopter and jetliner kill...,https://apnews.com/article/ronald-reagan-natio...,A midair collision between an Army helicopter ...,21 minutes ago,plane,https://dims.apnews.com/dims4/default/071c3d3/...,AP News
6,"Russian skating couple, world champions in 199...",https://www.reuters.com/world/us/renowned-russ...,Russian-born ice skating coaches and former wo...,8 hours ago,plane,None,Reuters
7,"Live updates: American Airlines plane, Black H...",https://www.cnn.com/us/live-news/plane-crash-d...,No survivors are expected after the midair col...,LIVE7 minutes ago,plane,https://media.cnn.com/api/v1/images/stellar/pr...,CNN
8,"'We will find out what happened,' NTSB vows af...",https://www.usatoday.com/story/news/nation/202...,"American Airlines Flight 5342 from Wichita, Ka...",LIVE49 minutes ago,plane,https://www.usatoday.com/gcdn/authoring/author...,USA Today
9,DC plane crash live updates: Dive teams ending...,https://abcnews.go.com/US/live-updates/reagan-...,An American Airlines regional jet went down in...,LIVE13 minutes ago,plane,https://i.abcnewsfe.com/a/0ec10b3a-67d7-439c-a...,"ABC News - Breaking News, Latest News and Videos"


In [44]:
from dotenv import load_dotenv
import os
load_dotenv()

from supabase import create_client, Client

supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))


In [45]:
# Convert DataFrame to records and upsert based on title and source as unique identifiers
supabase.table("Articles").upsert(
    df.to_dict(orient="records"),
    on_conflict=["title", "link"]
).execute()

APIResponse[~_ReturnT](data=[{'id': 1, 'title': 'American Airlines Plane Crash Tragedy', 'link': 'https://www.lcps.org/article/1997510', 'snippet': "Dear LCPS families and staff,. Our hearts are heavy as we process the devastating news of last night's tragic plane crash over the Potomac River involving...", 'time_scraped': '5 hours ago', 'source': 'Loudoun County Public Schools', 'created_at': '2025-01-31T03:19:00.58888+00:00', 'image_url': 'https://core-docs.s3.amazonaws.com/loudoun_county_public_schools_ar/article/image/large_5c06e76b-0724-4b41-a9d4-82bbdd6b7005.jpeg', 'query_term': 'plane'}, {'id': 2, 'title': 'Pilot of American Airlines jet that crashed near Washington DC had Georgia ties', 'link': 'https://www.fox5atlanta.com/news/pilot-american-airlines-jet-crashed-near-washington-dc-had-georgia-ties', 'snippet': 'A family with ties to Georgia is grieving after learning their loved one was one of the pilots killed in the crash between a small American Airlines plane...', 'time_sc